Activity:

(3 points) Using pandas, load the dataset and display the first 5 rows.

(3 points) Using SweetViz, do a quick EDA of the data set. In a markdown cell describe the main trends and patterns you see in the data.

(4 points) Define the input variables (X) and target variable (y). Split the data into training (80%) and testing (20%). 

(4 points) Scale the input variables.

(5 points) Build a baseline linear regressor and random forest regressor. Report both R2 and MSE for both models. Add a markdown cell discussing the performance of each of the models on the test data. 

(5 points) Use GridSearchCV to tune n_estimators, max_depth, min_samples_split and max_features. What is the best set of hyperparameters from your search?

(5 points) Examine the top 20 models (based on rank) from your GridSearchCV results. Add a markdown cell and discuss which set of hyperparameters you would choose and why. 

(3 points--if time) Compare the training versus test performance using a Random Forest model tuned with the best parameters from GridSearch CV. Is your model overfitting and how do you know?

Activity 1: Using pandas, load the dataset and display the first 5 rows.

In [7]:
pip install pandas numpy sweetviz scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------- ----- 7.1/8.1 MB 40.4 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 31.6 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- --


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np 
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [12]:
# loading the dataset and initial EDA with sweetviz designed to understand data
insurance = pd.read_csv(r'C:\Users\prchandr\Downloads\insurance.csv')

In [13]:
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Activity 2: Using SweetViz, do a quick EDA of the data set. In a markdown cell describe the main trends and patterns you see in the data.

In [14]:
report = sv.analyze(insurance)
report.show_html('insurance_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Activity 3: Define the input variables (X) and target variable (y). Split the data into training (80%) and testing (20%). 

In [21]:
# preparing data for modeling by encoding categorical variables and separating features and target variable
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [22]:
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

In [23]:
# splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Activity 4: Scale the input variables.

In [24]:
# scale the features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Activity 5: Build a baseline linear regressor and random forest regressor. Report both R2 and MSE for both models. Add a markdown cell discussing the performance of each of the models on the test data. 

In [25]:
# baseline linear regression model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

In [ ]:
# random forest regressor model
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f'Linear Regression MSE: {mse_lr}, R2: {r2_lr}')
print(f'Random Forest MSE: {mse_rf}, R2: {r2_rf}')

Linear Regression MSE: 33566439.73530044, R2: 0.7837888448800692
Random Forest MSE: 22179121.673664946, R2: 0.8571378569063584


The Random Forest model outperforms the Linear Regression model on the test data by having higher R^2 values and Mean Squared Error. The Random Forest has a lower mean squared error (22,179,121 vs. 33,566,439) and a higher (R^2) value (0.857 vs. 0.784), indicating that the random forest model captures about 85% of the data and has higher accuracy in predictions than the regression model.

Activity 6: Use GridSearchCV to tune n_estimators, max_depth, min_samples_split and max_features. What is the best set of hyperparameters from your search?

In [33]:
param_grid = {
    'n_estimators': [100, 200], 'max_depth': [None, 10, 20], 'min_samples_leaf': [2, 5], 'max_features': ['auto', 'sqrt']
}

In [35]:
# Use GridSearchCV to get cv_results_ (cross_val_score does not support 'ascending' or 'mean_test_score')
gs = GridSearchCV(
	estimator=RandomForestRegressor(random_state=42),
	param_grid=param_grid,
	cv=5,
	scoring='neg_mean_squared_error',
	return_train_score=True,
	n_jobs=-1
)
gs.fit(X_train, y_train)

cv_results = pd.DataFrame(gs.cv_results_)
cv_results = cv_results.sort_values('rank_test_score')
cv_results.head(20)

c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
60 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
41 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py", line 1329, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  F

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_leaf,param_n_estimators,params,split0_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
12,0.320051,0.015165,0.029337,0.001288,10,sqrt,2,100,"{'max_depth': 10, 'max_features': 'sqrt', 'min...",-2.506377e+07,...,-2.564797e+07,4.703429e+06,1,-1.260491e+07,-1.443126e+07,-1.242809e+07,-1.218784e+07,-1.291408e+07,-1.291324e+07,7.951160e+05
5,0.648327,0.030114,0.056484,0.004294,None,sqrt,2,200,"{'max_depth': None, 'max_features': 'sqrt', 'm...",-2.436794e+07,...,-2.582577e+07,5.073345e+06,2,-1.211971e+07,-1.278498e+07,-1.183793e+07,-1.144351e+07,-1.187076e+07,-1.201138e+07,4.432900e+05
21,0.709513,0.050337,0.054638,0.008426,20,sqrt,2,200,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",-2.436794e+07,...,-2.582577e+07,5.073345e+06,2,-1.211971e+07,-1.278498e+07,-1.183793e+07,-1.144351e+07,-1.187076e+07,-1.201138e+07,4.432900e+05
13,0.649786,0.019132,0.053825,0.006068,10,sqrt,2,200,"{'max_depth': 10, 'max_features': 'sqrt', 'min...",-2.514280e+07,...,-2.588405e+07,4.398623e+06,4,-1.267697e+07,-1.445755e+07,-1.248128e+07,-1.189399e+07,-1.264718e+07,-1.283139e+07,8.606370e+05
20,0.344034,0.021981,0.031478,0.001667,20,sqrt,2,100,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",-2.506601e+07,...,-2.625230e+07,5.049327e+06,5,-1.244422e+07,-1.295009e+07,-1.199720e+07,-1.162290e+07,-1.211006e+07,-1.222489e+07,4.475348e+05
4,0.380908,0.012904,0.029588,0.002827,None,sqrt,2,100,"{'max_depth': None, 'max_features': 'sqrt', 'm...",-2.506601e+07,...,-2.625230e+07,5.049327e+06,5,-1.244422e+07,-1.295009e+07,-1.199720e+07,-1.162290e+07,-1.211006e+07,-1.222489e+07,4.475348e+05
7,0.616401,0.028880,0.050604,0.001733,None,sqrt,5,200,"{'max_depth': None, 'max_features': 'sqrt', 'm...",-2.916878e+07,...,-2.944229e+07,3.665753e+06,7,-2.131747e+07,-2.309975e+07,-2.214430e+07,-2.000463e+07,-2.109252e+07,-2.153173e+07,1.039690e+06
23,0.639157,0.025730,0.053810,0.009579,20,sqrt,5,200,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",-2.916878e+07,...,-2.944229e+07,3.665753e+06,7,-2.131747e+07,-2.309975e+07,-2.214430e+07,-2.000463e+07,-2.109252e+07,-2.153173e+07,1.039690e+06
15,0.597417,0.036836,0.053244,0.003520,10,sqrt,5,200,"{'max_depth': 10, 'max_features': 'sqrt', 'min...",-2.929194e+07,...,-2.980172e+07,4.078905e+06,9,-2.145923e+07,-2.326593e+07,-2.278428e+07,-2.060186e+07,-2.161182e+07,-2.194462e+07,9.591919e+05
22,0.310401,0.028054,0.028253,0.001991,20,sqrt,5,100,"{'max_depth': 20, 'max_features': 'sqrt', 'min...",-3.005351e+07,...,-2.999655e+07,3.936902e+06,10,-2.159607e+07,-2.271940e+07,-2.383484e+07,-1.989452e+07,-2.167888e+07,-2.194474e+07,1.309062e+06


In [36]:
gs.best_params_

{'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'n_estimators': 100}

Activity 7: Examine the top 20 models (based on rank) from your GridSearchCV results. Add a markdown cell and discuss which set of hyperparameters you would choose and why.

I would choose a moderate max_depth like 10 to prevent the trees from becoming too complex and overfitting the training data. Setting min_samples_leaf = 2 helps ensure that each leaf has at least a small number of samples, which can improve generalization. I would use max_features = 'sqrt' because it is a common default for Random Forests and helps create diversity among trees. For n_estimators, I would choose n_estimators = 100 as it provides a sufficient number of trees for stable predictions without unnecessary computational cost.

Activity 8: Compare the training versus test performance using a Random Forest model tuned with the best parameters from GridSearch CV. Is your model overfitting and how do you know?

In [38]:
train_pred = gs.best_estimator_.predict(X_train)
test_pred = gs.best_estimator_.predict(X_test)

print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))
print("Train R2:", r2_score(y_train, train_pred))
print("Test R2:", r2_score(y_test, test_pred))

Train MSE: 13113362.211279586
Test MSE: 23461303.547066066
Train R2: 0.9091455566096527
Test R2: 0.8488789522948469


The mean squared error shows that the model has a lower average error on the training data (13,113,362) compared to the test data (23,461,303). Additionally, the R^2 value is higher for the training set (0.909) than for the test set (0.849).

This model is showing slight overfitting because it performs better on the training data than on unseen data based on the MSE and R^2 values from the training and test sets. However, the difference between the training and test performance is moderate, not extreme. The model still achieves a high (R^2) on the test set, meaning it generalizes well and maintains strong predictive accuracy.